In [0]:
# 1. Garante que alterações no arquivo .py sejam carregadas automaticamente sem reiniciar o cluster
%load_ext autoreload
%autoreload 2

import sys
import importlib
import funcoes_diversas.funcoes_util

# Força o Databricks a carregar as alterações recentes do arquivo .py
importlib.reload(funcoes_diversas.funcoes_util)

# 3. Importa as duas funções
from funcoes_diversas.funcoes_util import buscar_caminho, gerar_tabela_temporaria



In [0]:
mapa_criado = gerar_tabela_temporaria(["orders"])

In [0]:
df_product_silver = spark.sql("""with stocks as(
    select product_id, 
       sum(quantity) as total_stocks 
       from tmp_bronze_stocks 
       group by product_id)
    select p.product_id,
           p.product_name,
           b.brand_name,
           c.category_name,
           p.model_year,
           p.list_price,
           s.total_stocks
    from tmp_bronze_products        as p
    left join tmp_bronze_categories as c on p.category_id = c.category_id
    left join tmp_bronze_brands     as b on p.brand_id    = b.brand_id
    left join stocks                as s on p.product_id  = s.product_id """)

In [0]:
silver_path = buscar_caminho("silver")
# transformando consulta em uma tabela Silver 
df_product_silver.write\
    .mode("overwrite")\
    .format("delta")\
    .option("overwriteSchema", "true")\
    .save(f"{silver_path}/orders")